In [ ]:
# ==============================================================
# 🌾 Roboflow YOLOv11 → TensorFlow Lite (INT8) Export + Inference
# Target device: Raspberry Pi 4 B
# ==============================================================

# STEP 1️⃣ – Install dependencies
!pip install ultralytics roboflow -q

from ultralytics import YOLO
from roboflow import Roboflow
import os


In [ ]:
# STEP 2️⃣ – Connect to your Roboflow project
# 🔑 Replace with your own API key, workspace, and project name
rf = Roboflow(api_key="zU5zoCd2ANSr1c3NF8P8")      # e.g. "RF0KpN9Wn1QhvwC6Wm4M"
project = rf.workspace("walker-6w6cm").project("weed_detect_yolov11-mu5py")  # workspace + project
version = project.version(2)                         # dataset / model version number
dataset = version.download("yolov8")

loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to weed_detect_yolov11-2 in yolov8:: 100%|██████████| 2612/2612 [00:01<00:00, 2228.76it/s]


In [ ]:
print("✅ Dataset downloaded to:", dataset.location)

✅ Dataset downloaded to: /content/weed_detect_yolov11-2


In [ ]:
# STEP 3️⃣ – Initialize a lightweight YOLOv8 Nano model
# You can swap yolov8n.pt → yolov8s.pt for slightly higher accuracy
model = YOLO("yolov8n.pt")

In [ ]:
# STEP 4️⃣ – Train the model
model.train(
    data=f"{dataset.location}/data.yaml",  # path to data config
    epochs=90,          # increase if training stabilizes early
    imgsz=640,          # match dataset preprocessing
    batch=16,           # auto-adjust if Colab GPU memory low
    name="weed_detection_colab"
)

Ultralytics 8.3.228 🚀 Python-3.12.12 torch-2.8.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/weed_detect_yolov11-2/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=90, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=weed_detection_colab, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience=100, perspect

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0, 1])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7ca6c5e77c20>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
          0.04804

In [ ]:
# STEP 5️⃣ – Validate on the test/val split
metrics = model.val()
print("📊 Validation metrics:", metrics)

Ultralytics 8.3.228 🚀 Python-3.12.12 torch-2.8.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
Model summary (fused): 72 layers, 3,006,038 parameters, 0 gradients, 8.1 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1231.7±630.0 MB/s, size: 122.0 KB)
val: Scanning /content/weed_detect_yolov11-2/valid/labels.cache... 50 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 50/50 57.4Kit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 4/4 1.6it/s 2.5s
                   all         50        590      0.681      0.608      0.595      0.219
                  crop         39        269      0.624      0.548      0.502       0.14
                  weed         34        321      0.738      0.667      0.688      0.298
Speed: 9.9ms preprocess, 8.1ms inference, 0.0ms loss, 5.3ms postprocess per image
Results saved to /content/runs/detect/val
📊 Validation metrics: ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_

In [ ]:
# STEP 6️⃣ – Export a quantized TensorFlow Lite model
export_path = model.export(format="tflite", imgsz=416, int8=True)
print("✅ Export complete →", export_path)

Ultralytics 8.3.228 🚀 Python-3.12.12 torch-2.8.0+cu126 CPU (Intel Xeon CPU @ 2.00GHz)
WARNING ⚠️ INT8 export requires a missing 'data' arg for calibration. Using default 'data=coco8.yaml'.
💡 ProTip: Export to OpenVINO format for best performance on Intel hardware. Learn more at https://docs.ultralytics.com/integrations/openvino/

PyTorch: starting from '/content/runs/detect/weed_detection_colab/weights/best.pt' with input shape (1, 3, 416, 416) BCHW and output shape(s) (1, 6, 3549) (6.0 MB)
requirements: Ultralytics requirements ['sng4onnx>=1.0.1', 'onnx_graphsurgeon>=0.3.26', 'ai-edge-litert>=1.2.0', 'onnx>=1.12.0,<=1.19.1', 'onnx2tf>=1.26.3', 'onnxslim>=0.1.71', 'onnxruntime-gpu'] not found, attempting AutoUpdate...
Using Python 3.12.12 environment at: /usr
Resolved 20 packages in 3.77s
Prepared 11 packages in 5.26s
Installed 11 packages in 284ms
 + ai-edge-litert==2.0.3
 + backports-strenum==1.3.1
 + colorama==0.4.6
 + coloredlogs==15.0.1
 + humanfriendly==10.0
 + onnx==1.19.1
 + on

In [ ]:
# STEP 7️⃣ – Test inference with TFLite model (corrected path)
tflite_model = YOLO("/content/runs/detect/weed_detection_colab/weights/best_saved_model/best_int8.tflite")

results = tflite_model.predict(
    source=f"{dataset.location}/test/images",
    conf=0.25,
    save=True
)

print("✅ Inference complete — check 'runs/detect/predict' for annotated outputs.")


Loading /content/runs/detect/weed_detection_colab/weights/best_saved_model/best_int8.tflite for TensorFlow Lite inference...

image 1/50 /content/weed_detect_yolov11-2/test/images/1623872870546_frame_2672_jpg.rf.9ab72922dcee8944275af98804de080a.jpg: 416x416 6 crops, 66.0ms
image 2/50 /content/weed_detect_yolov11-2/test/images/1623872870546_frame_2715_jpg.rf.f70f0d78aee84180ce86225a86d863c0.jpg: 416x416 5 crops, 60.1ms
image 3/50 /content/weed_detect_yolov11-2/test/images/1623872870546_frame_2_jpg.rf.6b1f585cd5a8d379146f6bfe24c3e8a5.jpg: 416x416 5 crops, 60.8ms
image 4/50 /content/weed_detect_yolov11-2/test/images/1623872870546_frame_3108_jpg.rf.24e81c5cbc52cffe617451a709a4a389.jpg: 416x416 4 crops, 59.5ms
image 5/50 /content/weed_detect_yolov11-2/test/images/1624281034515_frame_129_jpg.rf.eacc796f6c5f26cc3469791afa4be7d7.jpg: 416x416 8 crops, 1 weed, 61.1ms
image 6/50 /content/weed_detect_yolov11-2/test/images/1624281034515_frame_4936_jpg.rf.6bec1f078f421f6cffe8bc8b62a6cb9e.jpg: 416x41

In [ ]:
!ls -lh /content/runs/detect/weed_detection_colab/weights/best_saved_model/


total 39M
drwxr-xr-x 2 root root 4.0K Nov 13 09:14 assets
-rw-r--r-- 1 root root 5.9M Nov 13 09:15 best_float16.tflite
-rw-r--r-- 1 root root  12M Nov 13 09:15 best_float32.tflite
-rw-r--r-- 1 root root 3.1M Nov 13 09:15 best_full_integer_quant.tflite
-rw-r--r-- 1 root root 3.1M Nov 13 09:15 best_int8.tflite
-rw-r--r-- 1 root root 3.1M Nov 13 09:15 best_integer_quant.tflite
-rw-r--r-- 1 root root   79 Nov 13 09:14 fingerprint.pb
-rw-r--r-- 1 root root  419 Nov 13 09:15 metadata.yaml
-rw-r--r-- 1 root root  12M Nov 13 09:14 saved_model.pb
drwxr-xr-x 2 root root 4.0K Nov 13 09:14 variables


In [ ]:
from google.colab import files
files.download("/content/runs/detect/weed_detection_colab/weights/best_saved_model/best_int8.tflite")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>